In [1]:
%pip install torch
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.metrics import(
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
from google.colab import files
uploaded = files.upload()


Saving speech_features.csv to speech_features.csv


In [3]:
df=pd.read_csv('speech_features.csv')
df.head()

,MFCC_Mean_1,MFCC_Mean_2,MFCC_Mean_3,MFCC_Mean_4,MFCC_Mean_5,MFCC_Mean_6,MFCC_Mean_7,MFCC_Mean_8,MFCC_Mean_9,MFCC_Mean_10,...,Contrast_STD_1,Contrast_STD_2,Contrast_STD_3,Contrast_STD_4,Contrast_STD_5,Contrast_STD_6,Contrast_STD_7,RollOff_Mean,RollOff_STD,Emotion
0,-337.198425,105.834122,4.703499,18.523153,8.154274,20.808151,-4.989314,4.450795,-5.468726,-2.746948,...,8.237610,4.069299,5.343327,4.287011,4.198801,5.156169,4.866263,9534.375000,7370.309353,neutral
1,-365.112823,97.251945,0.847367,17.629940,10.328798,19.593670,-6.221257,5.824074,-5.320165,-5.450251,...,9.067803,4.465693,5.762736,4.517117,4.036969,5.231014,5.068897,10442.557199,7396.500826,neutral
2,-397.269287,88.826836,4.091902,14.538698,7.572285,13.611512,-4.201235,5.421018,-6.086838,-3.690548,...,9.380212,4.029808,5.074805,4.697317,4.218557,5.008004,5.352342,11425.879727,8071.150947,neutral
3,-424.534821,74.249725,5.987121,14.322783,6.465185,13.501793,-2.348783,5.887998,-5.722302,-4.044089,...,9.997869,4.357622,5.330531,4.803650,4.164324,4.565001,5.390544,12561.715846,7630.444618,neutral
4,-380.204559,86.181702,6.601220,16.349792,7.120249,16.466585,-3.636997,5.079800,-6.108876,-0.968453,...,10.051774,4.739274,6.316195,5.337343,5.310067,5.239283,5.485773,11596.810567,7890.101509,calm


In [4]:
df.shape

(1440, 127)

In [5]:
df.columns

Index(['MFCC_Mean_1', 'MFCC_Mean_2', 'MFCC_Mean_3', 'MFCC_Mean_4',
       'MFCC_Mean_5', 'MFCC_Mean_6', 'MFCC_Mean_7', 'MFCC_Mean_8',
       'MFCC_Mean_9', 'MFCC_Mean_10',
       ...
       'Contrast_STD_1', 'Contrast_STD_2', 'Contrast_STD_3', 'Contrast_STD_4',
       'Contrast_STD_5', 'Contrast_STD_6', 'Contrast_STD_7', 'RollOff_Mean',
       'RollOff_STD', 'Emotion'],
      dtype='object', length=127)

In [6]:
df.isnull().sum()

,0
MFCC_Mean_1,0
MFCC_Mean_2,0
MFCC_Mean_3,0
MFCC_Mean_4,0
MFCC_Mean_5,0
...,...
Contrast_STD_6,0
Contrast_STD_7,0
RollOff_Mean,0
RollOff_STD,0


In [7]:
df['Emotion'].value_counts()

,count
Emotion,
calm,192
happy,192
sad,192
angry,192
disgust,192
fearful,192
surprised,192
neutral,96


In [8]:
X=df.drop('Emotion',axis=1)
y=df['Emotion']

In [9]:
le=LabelEncoder()
y=le.fit_transform(y)
print(le.classes_)
print(y[:10])

['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']
[5 5 5 5 1 1 1 1 1 1]


In [10]:
scaler=StandardScaler()
X=scaler.fit_transform(X)


In [11]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)


In [12]:
X_train_tensor=torch.tensor(X_train,dtype=torch.float32)
X_test_tensor=torch.tensor(X_test,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)

train_dataset=TensorDataset(
    X_train_tensor,
    y_train_tensor
)
test_dataset=TensorDataset(
    X_test_tensor,
    y_test_tensor
)

In [13]:
train_loader=DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
test_loader=DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [14]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device:{device}')
if torch.cuda.is_available():
  print(f'Device Name:{torch.cuda.get_device_name(0)}')

Using Device:cuda
Device Name:Tesla T4


In [35]:
class EmotionANN(nn.Module):
  def __init__(self,num_features,num_classes):
    super().__init__()
    self.network=nn.Sequential(
        nn.Linear(num_features,256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(256,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(64,num_classes)
    )
  def forward(self,x):
    return self.network(x)

In [36]:
num_features=X_train.shape[1]
num_classes=len(le.classes_)
model=EmotionANN(
    num_features=num_features,
    num_classes=num_classes
).to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=1e-3)


In [37]:
epochs=50
for epoch in range(epochs):
  model.train()
  running_loss=0
  for batch_features,batch_labels in train_loader:
    batch_features=batch_features.to(device,non_blocking=True)
    batch_labels=batch_labels.to(device,non_blocking=True)
    optimizer.zero_grad()
    outputs=model(batch_features)
    loss=criterion(outputs,batch_labels)
    loss.backward()
    optimizer.step()
    running_loss+=loss.item()
  epoch_loss=running_loss/len(train_loader)
  print(f'Epoch [{epoch+1}/{epochs}], Loss:{epoch_loss:.4f}')

Epoch [1/50], Loss:2.0731
Epoch [2/50], Loss:1.8835
Epoch [3/50], Loss:1.7567
Epoch [4/50], Loss:1.6740
Epoch [5/50], Loss:1.5780
Epoch [6/50], Loss:1.4761
Epoch [7/50], Loss:1.3761
Epoch [8/50], Loss:1.2905
Epoch [9/50], Loss:1.2039
Epoch [10/50], Loss:1.1189
Epoch [11/50], Loss:1.0904
Epoch [12/50], Loss:0.9887
Epoch [13/50], Loss:0.9435
Epoch [14/50], Loss:0.8597
Epoch [15/50], Loss:0.8219
Epoch [16/50], Loss:0.7806
Epoch [17/50], Loss:0.7162
Epoch [18/50], Loss:0.6956
Epoch [19/50], Loss:0.6525
Epoch [20/50], Loss:0.6238
Epoch [21/50], Loss:0.5745
Epoch [22/50], Loss:0.5397
Epoch [23/50], Loss:0.5050
Epoch [24/50], Loss:0.4963
Epoch [25/50], Loss:0.4703
Epoch [26/50], Loss:0.4622
Epoch [27/50], Loss:0.4160
Epoch [28/50], Loss:0.4086
Epoch [29/50], Loss:0.4027
Epoch [30/50], Loss:0.3663
Epoch [31/50], Loss:0.3615
Epoch [32/50], Loss:0.3670
Epoch [33/50], Loss:0.3410
Epoch [34/50], Loss:0.3309
Epoch [35/50], Loss:0.3170
Epoch [36/50], Loss:0.3286
Epoch [37/50], Loss:0.3025
Epoch [38/

In [38]:
model.eval()
actual=[]
predicted=[]

with torch.no_grad():
  for batch_features,batch_labels in test_loader:
    batch_features=batch_features.to(device,non_blocking=True)
    batch_labels=batch_labels.to(device,non_blocking=True)
    outputs=model(batch_features)
    predict=torch.argmax(outputs,dim=1)
    predicted.extend(predict.cpu().numpy())
    actual.extend(batch_labels.cpu().numpy())


In [40]:
print(f'Accuracy Score:{accuracy_score(actual,predicted)}')
print(f'Classification Report:\n{classification_report(actual,predicted)}')
print(f'Confusion Matrix:\n{confusion_matrix(actual,predicted)}')

Accuracy Score:0.75
Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.87      0.84        38
           1       0.74      0.92      0.82        38
           2       0.70      0.79      0.74        38
           3       0.68      0.69      0.68        39
           4       0.80      0.62      0.70        39
           5       0.57      0.63      0.60        19
           6       0.77      0.61      0.68        38
           7       0.89      0.82      0.85        39

    accuracy                           0.75       288
   macro avg       0.74      0.74      0.74       288
weighted avg       0.76      0.75      0.75       288

Confusion Matrix:
[[33  1  3  1  0  0  0  0]
 [ 0 35  1  0  0  1  1  0]
 [ 2  0 30  2  0  2  1  1]
 [ 2  3  2 27  3  0  2  0]
 [ 3  0  2  5 24  2  0  3]
 [ 0  5  0  0  0 12  2  0]
 [ 0  3  3  3  3  3 23  0]
 [ 1  0  2  2  0  1  1 32]]


In [41]:
print(f'Accuracy Score:{accuracy_score(actual,predicted)}')
print(f'Classification Report:\n{classification_report(actual,predicted)}')
print(f'Confusion Matrix:\n{confusion_matrix(actual,predicted)}')

Accuracy Score:0.75
Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.87      0.84        38
           1       0.74      0.92      0.82        38
           2       0.70      0.79      0.74        38
           3       0.68      0.69      0.68        39
           4       0.80      0.62      0.70        39
           5       0.57      0.63      0.60        19
           6       0.77      0.61      0.68        38
           7       0.89      0.82      0.85        39

    accuracy                           0.75       288
   macro avg       0.74      0.74      0.74       288
weighted avg       0.76      0.75      0.75       288

Confusion Matrix:
[[33  1  3  1  0  0  0  0]
 [ 0 35  1  0  0  1  1  0]
 [ 2  0 30  2  0  2  1  1]
 [ 2  3  2 27  3  0  2  0]
 [ 3  0  2  5 24  2  0  3]
 [ 0  5  0  0  0 12  2  0]
 [ 0  3  3  3  3  3 23  0]
 [ 1  0  2  2  0  1  1 32]]


In [42]:
#baseline ANNs were giving about 0.68, only batch normalization gave accuracy of 0.72 and finally with dropouts its giving accuracy of 0.75